# RAGAS Single-Turn Evaluation — End-to-End (Azure OpenAI)

A beginner-friendly, ready-to-run notebook for evaluating **single-turn** LLM / RAG
responses with [RAGAS](https://docs.ragas.io).

A *single-turn* evaluation looks at **one question and its one answer** at a time
(no back-and-forth conversation). For each example RAGAS can use up to four pieces
of information and gives you a score per metric.

**What's inside**
1. Install & configure RAGAS with Azure OpenAI
2. Understand the single-turn data format
3. The built-in ("defined") metrics, grouped into ready-to-use sets
   (`summarizer_metrics`, `rag_metrics`, `all_metrics`)
4. **Two ways to run an evaluation:**
   - **Method A — Excel upload (offline / bulk):** score a whole spreadsheet of
     questions & answers at once — start here
   - **Method B — Editable samples:** type your own examples directly in the
     notebook (great for a quick test)
5. **Custom metrics** with **Aspect Critic** (define your own pass/fail criteria)

**Prerequisites**
- Python 3.10+ (Google Colab works out of the box)
- An Azure OpenAI resource with a **GPT-5** deployment: endpoint, API key, deployment name, API version

> 💡 New here? Just run the cells top to bottom. Every section has a short note before the code.


## 1. Install dependencies

Run this once per session. If the packages are already installed you can skip it.


In [ ]:
%pip install \
    ragas==0.3.9 \
    langchain==0.3.27 \
    langchain-openai==0.3.28 \
    langchain-community==0.3.27 \
    langchain-huggingface==0.1.2 \
    openai==1.99.5 \
    datasets==4.0.0 \
    pandas==2.2.2 \
    sentence-transformers==3.4.1 \
    nest_asyncio==1.6.0 \
    openpyxl==3.1.5
print('Done. Skip this cell next time if packages are already installed.')

## 2. Imports

Core libraries used throughout the notebook. (Metric-specific imports live in their
own sections so it's clear what each part needs.)


In [ ]:
import os
import pandas as pd

from ragas import SingleTurnSample, EvaluationDataset, evaluate

# nest_asyncio lets RAGAS run its async calls inside a notebook / Colab cell.
import nest_asyncio
nest_asyncio.apply()

print('Imports ready')

## 3. Configure Azure OpenAI credentials

Fill in the four values below with your own Azure OpenAI details. The next cell
just confirms they were picked up (it never prints the key itself).


In [ ]:
import os

# 👇 Replace these with your Azure OpenAI values
os.environ["AZURE_OPENAI_API_KEY"]     = "YOUR_AZURE_OPENAI_API_KEY"
os.environ["AZURE_OPENAI_API_VERSION"] = "YOUR_AZURE_OPENAI_API_VERSION"
os.environ["AZURE_OPENAI_ENDPOINT"]    = "YOUR_AZURE_OPENAI_ENDPOINT"
os.environ["AZURE_DEPLOYMENT_NAME"]    = "YOUR_AZURE_DEPLOYMENT_NAME" 

In [ ]:
api_key     = os.environ.get('AZURE_OPENAI_API_KEY')     or os.environ.get('OPENAI_API_KEY')
api_version = os.environ.get('AZURE_OPENAI_API_VERSION') or os.environ.get('OPENAI_API_VERSION')
endpoint    = os.environ.get('AZURE_OPENAI_ENDPOINT')    or os.environ.get('OPENAI_API_BASE')
deployment  = os.environ.get('AZURE_DEPLOYMENT_NAME')    or os.environ.get('DEPLOYMENT_NAME')

print('API key present?', bool(api_key))
print('Endpoint:', endpoint)
print('Deployment:', deployment)

## 4. Initialize the LLM & embeddings for RAGAS

Most RAGAS metrics use an **LLM as a judge**, and a few (similarity-based) also need
an **embedding model**. Here we wrap Azure OpenAI as the judge LLM and a small local
HuggingFace model for embeddings.

### GPT-5 compatibility (important)

GPT-5 (and the o-series reasoning models) only accept **`temperature = 1`** — any
other value is rejected with:
*"Unsupported value: 'temperature' does not support 0.0 with this model. Only the
default (1) value is supported."*

RAGAS, however, drives the judge LLM at a low temperature (`0.01`, or `0.3` when it
asks for multiple samples), which makes those calls fail on GPT-5. Worse, in
RAGAS 0.3.x the `bypass_temperature=True` flag is only honoured on the **async**
code path (`agenerate_text`) — the **sync** path (`generate_text`) still forces the
low value. So we do two things:

1. create the Azure client with `temperature=1`, and
2. wrap it in a tiny subclass that pins `temperature=1` on **both** the sync and
   async paths.

> You may still see `DeprecationWarning`s about the wrapper classes — they are safe
> to ignore on RAGAS 0.3.x and the code works as-is.


In [ ]:
try:
    # AzureChatOpenAI now lives in langchain_openai (the langchain_community path is deprecated).
    from langchain_openai import AzureChatOpenAI
    from langchain_huggingface import HuggingFaceEmbeddings
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    use_langchain = True
except Exception as e:
    print('langchain_openai / langchain_huggingface not available. Error:', e)
    use_langchain = False

if use_langchain:
    # GPT-5 rejects any temperature other than 1. RAGAS otherwise forces a low
    # temperature on the judge LLM (0.01, or 0.3 when n > 1). This subclass pins
    # temperature=1 for BOTH generate_text (sync) and agenerate_text (async), so the
    # notebook works regardless of which RAGAS code path a metric happens to use.
    class GPT5LangchainLLMWrapper(LangchainLLMWrapper):
        def generate_text(self, prompt, n=1, temperature=None, stop=None, callbacks=None):
            return super().generate_text(prompt, n=n, temperature=1, stop=stop, callbacks=callbacks)

        async def agenerate_text(self, prompt, n=1, temperature=None, stop=None, callbacks=None):
            return await super().agenerate_text(prompt, n=n, temperature=1, stop=stop, callbacks=callbacks)

    # LangChain AzureChatOpenAI instance. `deployment` should point to a GPT-5 deployment.
    # temperature=1 is the only value GPT-5 accepts.
    llm_client = AzureChatOpenAI(
        azure_deployment=deployment,
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version=api_version,
        temperature=1,
    )
    # bypass_temperature=True stops the async path from lowering the temperature; the
    # subclass above additionally guarantees temperature=1 on the sync path.
    wrapped_llm = GPT5LangchainLLMWrapper(llm_client, bypass_temperature=True)
    print('Wrapped LangChain AzureChatOpenAI (GPT-5 compatible, temperature=1) for RAGAS')
else:
    # Fallback: wrap your own Azure client with the GPT-5-safe wrapper above.
    wrapped_llm = None
    print('No LLM wrapper created; set up your preferred Azure client and wrap it.')

# Embeddings used by similarity-based metrics (downloads a small model on first run)
evaluator_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
)
print('Embeddings ready')

## 5. The single-turn data format

Every example you evaluate is a `SingleTurnSample` built from up to four fields.
Not every metric needs all four — the table below shows what each field means.

| Field | Meaning | Example |
|---|---|---|
| `user_input` | the question / prompt | *"How many sick leave days are allowed each year?"* |
| `reference` | the ground-truth / expected answer | *"7 days of sick leave per year."* |
| `response` | the answer your model actually produced | *"Employees get 7 sick leave days each year."* |
| `retrieved_contexts` | the passage(s) your RAG system **actually retrieved** (a list) | *["Employees are entitled to 7 days of sick leave annually..."]* |

We use these exact field names everywhere below.


## 6. Built-in ("defined") metrics

These are the ready-made metrics this notebook uses. Some work **without any
retrieved context** and some **require it** — that's the key distinction below.

| Metric | What it measures | Needs | Needs context? |
|---|---|---|---|
| **AnswerCorrectness** | How well the response matches the reference (facts + meaning) | `response`, `reference` | No |
| **AnswerSimilarity** | Embedding similarity between response and reference | `response`, `reference` | No |
| **Faithfulness** | Is the response actually supported by the retrieved contexts? (no hallucination) | `response`, `retrieved_contexts` | Yes |
| **ContextRecall** | Did retrieval bring back the info needed to answer correctly? | `user_input`, `reference`, `retrieved_contexts` | Yes |
| **ContextPrecision** | Are the retrieved contexts relevant (and well-ranked) for the question? | `user_input`, `reference`, `retrieved_contexts` | Yes |
| **NoiseSensitivity** | How much does irrelevant retrieved context throw the answer off? | all four | Yes |

The cell below bundles these into **three ready-to-use metric sets** so you can pick
the one that matches the bot you're evaluating — no code changes, just swap the name:

| Metric set | Use it for | Contains |
|---|---|---|
| `summarizer_metrics` | Summarizer bots — Excel sheet has **no** context column | AnswerCorrectness, AnswerSimilarity |
| `rag_metrics` | RAG bots — Excel sheet **has** a retrieved-context column | all six metrics (everything a RAG eval needs) |
| `all_metrics` | Run absolutely everything | all six metrics |

> Reminder: the context column is the **actual retrieved context** your RAG system
> pulled up — not an "expected" context.


In [ ]:
from ragas.metrics import (
    AnswerCorrectness,
    AnswerSimilarity,
    Faithfulness,
    ContextRecall,
    ContextPrecision,
    NoiseSensitivity,
)

# Similarity metric needs the embedding model.
answer_similarity = AnswerSimilarity(embeddings=evaluator_embeddings)

# --- Metric sets ------------------------------------------------------------
# summarizer_metrics: NO retrieved context required. Use for summarizer bots whose
# Excel sheet has Question / Answer / Ground_Truth but no context column.
summarizer_metrics = [
    AnswerCorrectness(llm=wrapped_llm, answer_similarity=answer_similarity),
    answer_similarity,
]

# rag_metrics: the full set for RAG bots (includes the context-based metrics).
# Use when your Excel sheet DOES have a retrieved-context column.
rag_metrics = [
    AnswerCorrectness(llm=wrapped_llm, answer_similarity=answer_similarity),
    answer_similarity,
    Faithfulness(llm=wrapped_llm),
    ContextRecall(llm=wrapped_llm),
    ContextPrecision(llm=wrapped_llm),
    NoiseSensitivity(llm=wrapped_llm),
]

# all_metrics: everything (same coverage as rag_metrics) — a convenient alias.
all_metrics = rag_metrics

print('Metric sets ready:')
print('  summarizer_metrics ->', [getattr(m, "name", type(m).__name__) for m in summarizer_metrics])
print('  rag_metrics        ->', [getattr(m, "name", type(m).__name__) for m in rag_metrics])
print('  all_metrics        ->', [getattr(m, "name", type(m).__name__) for m in all_metrics])

## 7. Method A — Evaluate from an Excel file (offline / bulk)

**Start here for offline evaluation.** Each **row** = one example; each **column**
maps to a RAGAS field. This notebook is wired for the two bots you evaluate:

| Bot | Type | Has context column? | Metric set | Section |
|---|---|---|---|---|
| **EasyTravel Summarization Bot** | Summarizer | No | `summarizer_metrics` | 7.2 |
| **Policy Bot** | RAG | Yes (all fields) | `rag_metrics` (= `all_metrics`) | 7.3 |

**Expected columns** (generic names — case/spacing don't matter):

| Excel column | maps to RAGAS field | EasyTravel (summarizer) | Policy Bot (RAG) |
|---|---|:---:|:---:|
| `Question` | `user_input` | ✅ | ✅ |
| `Ground_Truth` | `reference` | ✅ | ✅ |
| `Answer` | `response` | ✅ | ✅ |
| `Contexts` | `retrieved_contexts` (the **actual retrieved** context) | — | ✅ |

- Run **7.1** once (the column-mapping helper), then run **7.2** for the EasyTravel
  Summarization Bot sheet and/or **7.3** for the Policy Bot sheet.
- **Multiple passages** in one `Contexts` cell? Separate them with `||` or new lines.


### 7.1 Column-mapping helper (run once)

Detects which of your columns match each RAGAS field, splits multi-passage context
cells, and skips blank cells. Used by both bot sections below.


In [ ]:
# Accepted header spellings for each RAGAS field (add your own if needed).
COLUMN_ALIASES = {
    "user_input":         ["question", "user_input", "input", "query", "prompt"],
    "reference":          ["ground_truth", "groundtruth", "reference", "expected_output", "expected", "gold_answer"],
    "response":           ["answer", "response", "actual_output", "output", "prediction", "model_answer"],
    "retrieved_contexts": ["contexts", "context", "retrieved_contexts", "retrieval_context", "retrieved_context"],
}
CONTEXT_DELIMITERS = ["||", "\n", ";"]   # how multiple passages in one cell may be separated

def _norm(name):
    return str(name).strip().lower().replace(" ", "_")

def build_field_map(columns):
    norm_cols = {_norm(c): c for c in columns}
    field_map = {}
    for field, aliases in COLUMN_ALIASES.items():
        for alias in aliases:
            if alias in norm_cols:
                field_map[field] = norm_cols[alias]
                break
    return field_map

def _is_blank(value):
    return value is None or (isinstance(value, float) and pd.isna(value)) or str(value).strip() == ""

def split_contexts(value):
    if _is_blank(value):
        return None
    text = str(value).strip()
    for delim in CONTEXT_DELIMITERS:
        if delim in text:
            parts = [p.strip() for p in text.split(delim) if p.strip()]
            return parts or None
    return [text]

def excel_to_dataset(df):
    field_map = build_field_map(df.columns)
    present = set(field_map.keys())

    print('Detected column -> RAGAS field:')
    for field, col in field_map.items():
        print(f'   {col!r}  ->  {field}')
    missing = set(COLUMN_ALIASES) - present
    if missing:
        print('\nColumns not present:', ', '.join(sorted(missing)))
        print('   -> fine for a summarizer sheet (no context); use summarizer_metrics.')

    samples = []
    for _, row in df.iterrows():
        kwargs = {}
        for field, col in field_map.items():
            value = row.get(col)
            if field == "retrieved_contexts":
                ctx = split_contexts(value)
                if ctx is not None:
                    kwargs[field] = ctx
            elif not _is_blank(value):
                kwargs[field] = str(value).strip()
        if kwargs:                      # skip completely empty rows
            samples.append(SingleTurnSample(**kwargs))

    print(f'\nBuilt {len(samples)} sample(s) from {len(df)} row(s).')
    return EvaluationDataset(samples=samples), present

print('Helper ready. Now run 7.2 (EasyTravel) and/or 7.3 (Policy Bot).')

### 7.2 EasyTravel Summarization Bot (no context)

Upload the **EasyTravel Summarization Bot** spreadsheet — it has Question, Answer and
Ground_Truth but **no** context column. It's scored with `summarizer_metrics`
(AnswerCorrectness + AnswerSimilarity), so the run finishes cleanly with no
missing-context warnings or errors.


In [ ]:
from google.colab import files

print('Upload the EasyTravel Summarization Bot Excel file (.xlsx)')
easytravel_upload = files.upload()                       # choose the EasyTravel .xlsx
easytravel_filename = next(iter(easytravel_upload))
easytravel_df = pd.read_excel(easytravel_filename)
print(f'Loaded "{easytravel_filename}" with {len(easytravel_df)} row(s).')
print('Columns found:', list(easytravel_df.columns))

easytravel_dataset, _ = excel_to_dataset(easytravel_df)

easytravel_results = evaluate(
    dataset=easytravel_dataset,
    metrics=summarizer_metrics,     # no-context metrics for the summarizer bot
    llm=wrapped_llm,
    embeddings=evaluator_embeddings,
)
easytravel_scores_df = easytravel_results.to_pandas()
print(easytravel_scores_df.to_string())

easytravel_scores_df.to_excel('easytravel_summarization_results.xlsx', index=False)
print('\nSaved scores to easytravel_summarization_results.xlsx')

### 7.3 Policy Bot (RAG — all fields)

Upload the **Policy Bot** spreadsheet — a RAG application, so it has every field
including `Contexts`. Because all fields are present it runs the full `rag_metrics`
set (identical to `all_metrics`): AnswerCorrectness, AnswerSimilarity, Faithfulness,
ContextRecall, ContextPrecision and NoiseSensitivity.


In [ ]:
from google.colab import files

print('Upload the Policy Bot (RAG) Excel file (.xlsx)')
policy_upload = files.upload()                           # choose the Policy Bot .xlsx
policy_filename = next(iter(policy_upload))
policy_df = pd.read_excel(policy_filename)
print(f'Loaded "{policy_filename}" with {len(policy_df)} row(s).')
print('Columns found:', list(policy_df.columns))

policy_dataset, _ = excel_to_dataset(policy_df)

policy_results = evaluate(
    dataset=policy_dataset,
    metrics=rag_metrics,            # full RAG set (uses the Contexts column)
    llm=wrapped_llm,
    embeddings=evaluator_embeddings,
)
policy_scores_df = policy_results.to_pandas()
print(policy_scores_df.to_string())

policy_scores_df.to_excel('policy_bot_rag_results.xlsx', index=False)
print('\nSaved scores to policy_bot_rag_results.xlsx')

## 8. Method B — Evaluate your own editable samples

Best for a quick check. The dictionaries below are **placeholders** — edit the text
to match an example you care about, then run the cells. Everything stays in the
notebook; no file needed.


### 8.1 Placeholder examples (edit these)

Two ready-to-edit cases: one where the answer is good, and one where it's clearly
wrong. Change the strings to your own content.


In [ ]:
# A well-answered case
sample_case = {
    "input": "How much maternity leave are female employees entitled to?",
    "expected_output": "Female employees are entitled to 26 weeks of paid maternity leave as per the Maternity Benefit Act.",
    "actual_output": "Female employees receive 26 weeks of paid maternity leave, which can start up to 8 weeks before delivery.",
    "retrieval_context": [
        "Maternity leave provides 26 weeks of paid time off for female employees.",
        "Up to 8 weeks can be taken before the expected date of delivery.",
    ],
}

# A mismatched case: the answer does not match the question/reference
mismatched_case = {
    "input": "What should employees wear on casual Fridays?",
    "expected_output": "Employees may wear clean jeans and simple t-shirts without rips or offensive prints.",
    "actual_output": "Employees can take one extra leave day on Fridays as part of work-life balance initiatives.",
    "retrieval_context": [
        "Casual Fridays allow jeans and casual shoes, provided clothing is clean and appropriate.",
        "Slogans or political prints are not permitted on t-shirts.",
    ],
}

### 8.2 Build samples and run the built-in metrics

Here we turn three editable examples into `SingleTurnSample`s, bundle them into an
`EvaluationDataset`, and score them with `all_metrics` from Section 6 (the samples
include a context field, so every metric can run).


In [ ]:
# Edit these three samples freely — add or remove as you like.
sample1 = SingleTurnSample(
    user_input="What can employees wear on casual Fridays?",
    retrieved_contexts=["Clean jeans and plain t-shirts are allowed; no rips or offensive prints."],
    response="Employees may wear neat jeans and simple t-shirts on casual Fridays.",
    reference="Clean jeans and plain t-shirts without rips or prints.",
)
sample2 = SingleTurnSample(
    user_input="How many sick leave days are allowed each year?",
    retrieved_contexts=["Employees are entitled to 7 days of sick leave annually with a medical note if absent over 3 days."],
    response="Employees get 7 sick leave days each year, with a doctor's note if more than 3 days.",
    reference="7 days of sick leave per year.",
)
sample3 = SingleTurnSample(
    user_input="Can employees book travel independently?",
    retrieved_contexts=["Travel must be booked through approved vendors unless written approval is given."],
    response="Employees can book travel only with prior written approval from the travel coordinator.",
    reference="Only with prior written approval from the travel coordinator or finance.",
)

dataset = EvaluationDataset(samples=[sample1, sample2, sample3])

response = evaluate(dataset=dataset, metrics=all_metrics, llm=wrapped_llm, embeddings=evaluator_embeddings)
print(response.to_pandas().to_string())

### 8.3 (Optional) Score a single placeholder case

You can also evaluate just one of the dictionaries above. This reuses `sample_case`
and a smaller metric list.


In [ ]:
single_df = pd.DataFrame([{
    'user_input': sample_case['input'],
    'response': sample_case['actual_output'],
    'retrieved_contexts': sample_case['retrieval_context'],
    'reference': sample_case['expected_output'],
}])
single_dataset = EvaluationDataset.from_pandas(single_df)

single_metrics = [
    AnswerCorrectness(llm=wrapped_llm, answer_similarity=answer_similarity),
    Faithfulness(llm=wrapped_llm),
    ContextRecall(llm=wrapped_llm),
]

try:
    eval_result = evaluate(dataset=single_dataset, metrics=single_metrics,
                           llm=wrapped_llm, embeddings=evaluator_embeddings)
    print('Evaluate() returned:')
    print(eval_result)
except Exception as e:
    print('evaluate(...) failed:', e)
    print('Ensure the ragas.evaluate signature matches your installed version.')

## 9. Custom metrics with Aspect Critic

Sometimes you want to check something the built-in metrics don't cover — e.g.
*"is the answer polite?"* or *"is it concise?"*. **Aspect Critic** lets you describe
a yes/no criterion in plain English, and the LLM judges it (returns **1 = pass**,
**0 = fail**).

Below are five custom criteria — adapt the `definition` text to whatever you need.


In [ ]:
from ragas.metrics import AspectCritic
from ragas.dataset_schema import SingleTurnSample

answer_correctness_critic = AspectCritic(
    name="Answer Correctness",
    definition="Does the answer correctly convey the same meaning as the reference answer?",
    llm=wrapped_llm,
)

fluency = AspectCritic(
    name="Fluency",
    definition="Is the answer grammatically correct and easy to understand?",
    llm=wrapped_llm,
)

completeness = AspectCritic(
    name="Completeness",
    definition="Does the answer fully address the question using the provided context?",
    llm=wrapped_llm,
)

conciseness = AspectCritic(
    name="Conciseness",
    definition="Is the answer to the point, without unnecessary or repeated information?",
    llm=wrapped_llm,
)

groundedness = AspectCritic(
    name="Groundedness",
    definition="Is every claim in the answer supported by the retrieved context, with nothing made up?",
    llm=wrapped_llm,
)

custom_metrics = [answer_correctness_critic, fluency, completeness, conciseness, groundedness]
print('Custom metrics ready:', [m.name for m in custom_metrics])

### 9.1 Run custom metrics on a single sample

Uses the `mismatched_case` from Section 8 so you can see how a wrong answer scores.


In [ ]:
sample = SingleTurnSample(
    user_input=mismatched_case['input'],
    retrieved_contexts=mismatched_case['retrieval_context'],
    response=mismatched_case['actual_output'],
    reference=mismatched_case['expected_output'],
)

async def run_custom_metrics_scores_only():
    print('Sample:')
    print(f'  User Input: {sample.user_input}')
    print(f'  Response:   {sample.response}')
    print(f'  Reference:  {sample.reference}')
    print('\n  Metric Scores (1 = pass, 0 = fail):')
    for metric in custom_metrics:
        score = await metric.single_turn_ascore(sample)
        print(f'    {metric.name}: {score}')

await run_custom_metrics_scores_only()

### 9.2 (Optional) Run custom metrics across a whole dataset

Aspect Critic metrics also work with `evaluate()`, so you can score every row of the
sample dataset (or swap in `excel_dataset`) at once.


In [ ]:
custom_results = evaluate(
    dataset=dataset,            # the editable samples; swap in easytravel_dataset or policy_dataset for the Excel rows
    metrics=custom_metrics,
    llm=wrapped_llm,
)
print(custom_results.to_pandas().to_string())

## 10. Recap & next steps

You now have a single-turn evaluation notebook that can:
- score the **EasyTravel Summarization Bot** sheet offline with `summarizer_metrics`
  (Section 7.2) and the **Policy Bot** RAG sheet with `rag_metrics` (Section 7.3),
- score **your own typed examples** for a quick check (Method B),
- using both **built-in metrics** and **custom Aspect Critic** criteria.

**Where to go next**
- Add more rows to either bot's Excel file and re-run its section (7.2 or 7.3).
- New summarizer bot with no context? Reuse Section 7.2 with `summarizer_metrics`.
  New RAG bot with a context column? Reuse Section 7.3 with `rag_metrics`.
- Write new Aspect Critic definitions in Section 9 for criteria specific to your use case.
- Compare scores across model versions by changing the Azure deployment in Section 3.

📚 RAGAS docs: https://docs.ragas.io
